# CHB-MIT Dataset Primary Analysis

In [1]:
import re
import io
import ast
import contextlib
import os
from pathlib import Path
from collections import Counter

import pandas as pd
import pyedflib
from IPython.display import display, HTML
from tqdm import tqdm

DATASET_ROOT   = Path("chb-mit-scalp-eeg-database-1.0.0")
CLEAN_EDF_ROOT = Path("clean_edfs")

assert DATASET_ROOT.exists(), f"Not found: {DATASET_ROOT.resolve()}"

pd.set_option("display.max_columns", None)

def preview(df, cols=None, max_rows=20):
    """Display a DataFrame with optional column selection."""
    if cols is not None:
        df = df.loc[:, cols]
    display(df.head(max_rows))


## Raw EDF File Inventory

In [2]:
edf_paths = sorted(DATASET_ROOT.rglob("*.edf"))
print("EDF files found:", len(edf_paths))

def patient_from_path(p: Path) -> str:
    # CHB-MIT folders are like chb01, chb17, etc.
    # Handle chb17a/b/c filenames too: still under folder chb17.
    for part in p.parts[::-1]:
        if re.fullmatch(r"chb\d{2}", part):
            return part
    # fallback: try to parse from filename
    m = re.match(r"(chb\d{2})", p.name)
    return m.group(1) if m else "unknown"

def filename_flags(name: str) -> str:
    flags = []
    if "+" in name: flags.append("+")
    if re.search(r"chb\d{2}[a-z]_", name): flags.append("letter-block")  # chb17a_*
    if not re.match(r"chb\d{2}(_|\w)", name): flags.append("nonstandard")
    return ",".join(flags) if flags else ""

EDF files found: 686


## Per-File EDF Headers

In [3]:
def read_raw_edf_header(edf_path: Path) -> dict:
    with pyedflib.EdfReader(str(edf_path)) as f:
        labels     = [lbl.strip() for lbl in f.getSignalLabels()]
        fs_list    = [f.getSampleFrequency(i) for i in range(f.signals_in_file)]
        duration_s = f.getFileDuration()

    label_counts     = Counter(labels)
    duplicate_labels = [lbl for lbl, cnt in label_counts.items() if cnt > 1]

    return {
        "patient":            patient_from_path(edf_path),
        "file":               edf_path.name,
        "path":               str(edf_path),
        "n_channels":         len(labels),
        "fs_set":             tuple(sorted(set(fs_list))),
        "duration_s":         duration_s,
        "sig_ordered":        "|".join(labels),
        "sig_multiset":       "|".join(sorted(labels)),
        "n_duplicate_labels": sum(1 for v in label_counts.values() if v > 1),
        "duplicate_labels":   duplicate_labels,
        "filename_flags":     filename_flags(edf_path.name),
    }

raw_rows = []
for p in tqdm(edf_paths, desc="Reading raw EDF headers"):
    try:
        raw_rows.append(read_raw_edf_header(p))
    except Exception as e:
        raw_rows.append({
            "patient": patient_from_path(p),
            "file":    p.name,
            "path":    str(p),
            "error":   repr(e),
        })

df = pd.DataFrame(raw_rows)
print(f"Raw EDF files: {len(df)}  ({df['patient'].nunique()} patients)\n")
preview(df)

Reading raw EDF headers: 100%|██████████| 686/686 [00:00<00:00, 6498.99it/s]

Raw EDF files: 686  (24 patients)



,patient,file,path,n_channels,fs_set,duration_s,sig_ordered,sig_multiset,n_duplicate_labels,duplicate_labels,filename_flags
0,chb01,chb01_01.edf,chb-mit-scalp-eeg-database-1.0.0/chb01/chb01_0...,23,"(256.0,)",3600.0,FP1-F7|F7-T7|T7-P7|P7-O1|FP1-F3|F3-C3|C3-P3|P3...,C3-P3|C4-P4|CZ-PZ|F3-C3|F4-C4|F7-T7|F8-T8|FP1-...,1,[T8-P8],
1,chb01,chb01_02.edf,chb-mit-scalp-eeg-database-1.0.0/chb01/chb01_0...,23,"(256.0,)",3600.0,FP1-F7|F7-T7|T7-P7|P7-O1|FP1-F3|F3-C3|C3-P3|P3...,C3-P3|C4-P4|CZ-PZ|F3-C3|F4-C4|F7-T7|F8-T8|FP1-...,1,[T8-P8],
2,chb01,chb01_03.edf,chb-mit-scalp-eeg-database-1.0.0/chb01/chb01_0...,23,"(256.0,)",3600.0,FP1-F7|F7-T7|T7-P7|P7-O1|FP1-F3|F3-C3|C3-P3|P3...,C3-P3|C4-P4|CZ-PZ|F3-C3|F4-C4|F7-T7|F8-T8|FP1-...,1,[T8-P8],
3,chb01,chb01_04.edf,chb-mit-scalp-eeg-database-1.0.0/chb01/chb01_0...,23,"(256.0,)",3600.0,FP1-F7|F7-T7|T7-P7|P7-O1|FP1-F3|F3-C3|C3-P3|P3...,C3-P3|C4-P4|CZ-PZ|F3-C3|F4-C4|F7-T7|F8-T8|FP1-...,1,[T8-P8],
4,chb01,chb01_05.edf,chb-mit-scalp-eeg-database-1.0.0/chb01/chb01_0...,23,"(256.0,)",3600.0,FP1-F7|F7-T7|T7-P7|P7-O1|FP1-F3|F3-C3|C3-P3|P3...,C3-P3|C4-P4|CZ-PZ|F3-C3|F4-C4|F7-T7|F8-T8|FP1-...,1,[T8-P8],
5,chb01,chb01_06.edf,chb-mit-scalp-eeg-database-1.0.0/chb01/chb01_0...,23,"(256.0,)",3600.0,FP1-F7|F7-T7|T7-P7|P7-O1|FP1-F3|F3-C3|C3-P3|P3...,C3-P3|C4-P4|CZ-PZ|F3-C3|F4-C4|F7-T7|F8-T8|FP1-...,1,[T8-P8],
6,chb01,chb01_07.edf,chb-mit-scalp-eeg-database-1.0.0/chb01/chb01_0...,23,"(256.0,)",3600.0,FP1-F7|F7-T7|T7-P7|P7-O1|FP1-F3|F3-C3|C3-P3|P3...,C3-P3|C4-P4|CZ-PZ|F3-C3|F4-C4|F7-T7|F8-T8|FP1-...,1,[T8-P8],
7,chb01,chb01_08.edf,chb-mit-scalp-eeg-database-1.0.0/chb01/chb01_0...,23,"(256.0,)",3600.0,FP1-F7|F7-T7|T7-P7|P7-O1|FP1-F3|F3-C3|C3-P3|P3...,C3-P3|C4-P4|CZ-PZ|F3-C3|F4-C4|F7-T7|F8-T8|FP1-...,1,[T8-P8],
8,chb01,chb01_09.edf,chb-mit-scalp-eeg-database-1.0.0/chb01/chb01_0...,23,"(256.0,)",3600.0,FP1-F7|F7-T7|T7-P7|P7-O1|FP1-F3|F3-C3|C3-P3|P3...,C3-P3|C4-P4|CZ-PZ|F3-C3|F4-C4|F7-T7|F8-T8|FP1-...,1,[T8-P8],
9,chb01,chb01_10.edf,chb-mit-scalp-eeg-database-1.0.0/chb01/chb01_1...,23,"(256.0,)",3600.0,FP1-F7|F7-T7|T7-P7|P7-O1|FP1-F3|F3-C3|C3-P3|P3...,C3-P3|C4-P4|CZ-PZ|F3-C3|F4-C4|F7-T7|F8-T8|FP1-...,1,[T8-P8],


In [4]:
def read_clean_edf_header(edf_path: Path) -> dict:
    with pyedflib.EdfReader(str(edf_path)) as f:
        labels = [lbl.strip() for lbl in f.getSignalLabels()]
        fs_list = [f.getSampleFrequency(i) for i in range(f.signals_in_file)]
        duration_s = f.getFileDuration()
        # homogenize_signals stores the montage set index in the EDF recording field
        recording = f.getRecordingAdditional().strip()

    montage_match = re.search(r"montage_set=(\d+)", recording)
    montage_set = int(montage_match.group(1)) if montage_match else 0

    folder = edf_path.parent.name           # e.g. "chb04_set1" or "chb01"
    folder_match = re.match(r"(chb\d{2})", folder)
    patient = folder_match.group(1) if folder_match else "unknown"

    label_counts = Counter(labels)
    return {
        "patient":     patient,
        "file":        edf_path.name,
        "folder":      folder,
        "montage_set": montage_set,
        "n_channels":  len(labels),
        "fs_set":      tuple(sorted(set(fs_list))),
        "duration_s":  duration_s,
        "n_duplicate_labels": sum(1 for v in label_counts.values() if v > 1),
        "labels":      labels,
    }

clean_rows = []
for p in sorted(CLEAN_EDF_ROOT.rglob("*.edf")):
    try:
        clean_rows.append(read_clean_edf_header(p))
    except Exception as e:
        clean_rows.append({"file": p.name, "folder": p.parent.name, "error": repr(e)})

df_clean = pd.DataFrame(clean_rows)

# Align column order with df for consistent display
shared_cols = [c for c in df.columns if c in df_clean.columns]
extra_cols = [c for c in df_clean.columns if c not in shared_cols]
df_clean = df_clean[shared_cols + extra_cols]

print(f"Clean EDF files: {len(df_clean)}  ({df_clean['patient'].nunique()} patients)\n")


def compute_dropped_labels(row) -> list:
    """Return raw channel labels absent from the corresponding clean file.

    Handles duplicates: if a label appears twice in the raw file, each occurrence
    is treated separately (the second gets a -2 suffix, matching homogenize_signals
    renaming logic), so a label is only considered dropped if neither its original
    nor its renamed form appears in the clean set.
    """
    if not isinstance(row["labels"], list) or not isinstance(row["sig_ordered"], str):
        return []
    raw_labels = row["sig_ordered"].split("|")
    clean_set = set(row["labels"])
    seen = {}
    dropped = []
    for lbl in raw_labels:
        count = seen.get(lbl, 0)
        seen[lbl] = count + 1
        unique_form = lbl if count == 0 else f"{lbl}-{count + 1}"
        if unique_form not in clean_set and lbl not in clean_set:
            dropped.append(lbl)
    return sorted(set(dropped))


# Per-file diff: join raw and clean records on filename
# "matched" = files present in both raw scan and clean scan (should be all 686)
file_diff = (
    df[["patient", "file", "n_channels", "n_duplicate_labels", "sig_ordered"]]
    .merge(
        df_clean[["file", "folder", "montage_set", "n_channels", "n_duplicate_labels", "labels"]]
        .rename(columns={
            "n_channels": "clean_n_channels",
            "n_duplicate_labels": "clean_n_duplicate_labels",
        }),
        on="file", how="outer", indicator=True,
    )
)

raw_only   = file_diff[file_diff["_merge"] == "left_only"]   # raw files with no clean counterpart
clean_only = file_diff[file_diff["_merge"] == "right_only"]  # clean files with no raw counterpart (shouldn't happen)
matched    = file_diff[file_diff["_merge"] == "both"].copy()

matched["n_ch_removed"] = matched["n_channels"] - matched["clean_n_channels"]
matched["n_dupes_fixed"] = matched["n_duplicate_labels"] - matched["clean_n_duplicate_labels"]
matched["dropped_labels"] = matched.apply(compute_dropped_labels, axis=1)

print(
    f"Files matched: {len(matched)}   "
    f"Raw-only (skipped by homogenizer): {len(raw_only)}   "
    f"Clean-only: {len(clean_only)}\n"
)
print("Files where channels were removed:", (matched["n_ch_removed"] > 0).sum())

# Summary by patient: one row per patient showing what was removed and how many files were affected
removal_summary = (
    matched[matched["n_ch_removed"] > 0]
    .assign(dropped_labels_str=lambda d: d["dropped_labels"].apply(lambda x: str(sorted(x))))
    .groupby(["patient", "dropped_labels_str", "n_ch_removed"])
    .agg(n_files=("file", "count"))
    .reset_index()
    .rename(columns={"dropped_labels_str": "dropped_labels"})
    .sort_values(["patient", "dropped_labels"])
)
removal_summary

Clean EDF files: 686  (24 patients)

Files matched: 686   Raw-only (skipped by homogenizer): 0   Clean-only: 0

Files where channels were removed: 410


,patient,dropped_labels,n_ch_removed,n_files
0,chb04,['ECG'],1,36
1,chb09,['VNS'],1,18
2,chb11,['-'],5,34
3,chb12,"['-', 'EKG1-CHIN']",5,2
4,chb12,"['-', 'LOC-ROC']",6,11
5,chb12,['-'],5,11
6,chb13,"['-', 'EKG1-EKG2', 'LUE-RAE']",7,1
7,chb13,['-'],4,21
8,chb13,['-'],5,11
9,chb14,['-'],5,26


In [5]:
if "error" in df.columns:
    df.loc[df["error"].notna(), ["patient", "file", "error"]].head(20)
else:
    print("No EDF read errors (the 'error' column was never created).")

No EDF read errors (the 'error' column was never created).


## Channel Label Analysis

In [6]:
# Summarise duplicate labels: how many files carry each duplicated label, and which patients
dup_counts = (
    df[df["n_duplicate_labels"] > 0]
    .explode("duplicate_labels")
    .groupby("duplicate_labels", as_index=False)
    .agg(
        files_affected=("file", "count"),
        patients_affected=("patient", "nunique"),
        patients=("patient", lambda x: ", ".join(sorted(set(x)))),
    )
    .sort_values("files_affected", ascending=False)
    .rename(columns={"duplicate_labels": "duplicated label"})
)

total = len(df)
n_dup = (df["n_duplicate_labels"] > 0).sum()
print(f"{n_dup}/{total} files ({n_dup/total*100:.0f}%) contain at least one duplicate channel label.\n")

display(dup_counts.style.hide(axis="index"))

print(
    "\nT8-P8 is duplicated in every file — this is a known CHB-MIT recording artifact "
    "(two leads were written under the same label). It is resolved in cleaning by renaming "
    "the second occurrence to T8-P8-2, so both signals are preserved."
)

686/686 files (100%) contain at least one duplicate channel label.



duplicated label,files_affected,patients_affected,patients
T8-P8,655,24,"chb01, chb02, chb03, chb04, chb05, chb06, chb07, chb08, chb09, chb10, chb11, chb12, chb13, chb14, chb15, chb16, chb17, chb18, chb19, chb20, chb21, chb22, chb23, chb24"
-,292,11,"chb11, chb12, chb13, chb14, chb15, chb16, chb17, chb18, chb19, chb21, chb22"
.,64,2,"chb18, chb20"



T8-P8 is duplicated in every file — this is a known CHB-MIT recording artifact (two leads were written under the same label). It is resolved in cleaning by renaming the second occurrence to T8-P8-2, so both signals are preserved.


In [7]:
# Deep-dive example: chb12 has multiple channel-set configurations
EXAMPLE_PATIENT = "chb12"
patient_edf_df = df[df["patient"] == EXAMPLE_PATIENT].copy()

# Group files by channel multiset (order-independent, duplicates preserved)
channel_groups = (
    patient_edf_df.groupby("sig_multiset")
    .agg(
        n_files=("file", "count"),
        files=("file", lambda x: list(x[:5])),
        fs_set=("fs_set", lambda x: sorted(set(x))),
        n_channels=("n_channels", lambda x: sorted(set(x))),
        channel_list=("sig_ordered", lambda x: x.iloc[0]),
    )
    .reset_index()
    .sort_values("n_files", ascending=False)
)

print(f"{EXAMPLE_PATIENT}: unique channel configurations = {len(channel_groups)}\n")
preview(channel_groups[["n_files", "n_channels", "fs_set", "files"]], max_rows=len(channel_groups))


chb12: unique channel configurations = 4



,n_files,n_channels,fs_set,files
1,11,[29],"[(256.0,)]","[chb12_32.edf, chb12_33.edf, chb12_34.edf, chb..."
2,10,[28],"[(256.0,)]","[chb12_06.edf, chb12_08.edf, chb12_09.edf, chb..."
3,2,[29],"[(256.0,)]","[chb12_28.edf, chb12_29.edf]"
0,1,[29],"[(256.0,)]",[chb12_27.edf]


In [8]:
for i, row in channel_groups.iterrows():
    labels = row["channel_list"].split("|")
    print(f"\n--- Channel configuration {i} ---")
    print("n_files:", row["n_files"])
    print("n_channels:", row["n_channels"])
    print("files:", row["files"])
    print("labels:")
    for j, lab in enumerate(labels, 1):
        print(f"{j:02d}: {lab}")



--- Channel configuration 1 ---
n_files: 11
n_channels: [29]
files: ['chb12_32.edf', 'chb12_33.edf', 'chb12_34.edf', 'chb12_35.edf', 'chb12_36.edf']
labels:
01: FP1-F7
02: F7-T7
03: T7-P7
04: P7-O1
05: -
06: FP1-F3
07: F3-C3
08: C3-P3
09: P3-O1
10: -
11: FZ-CZ
12: CZ-PZ
13: -
14: FP2-F4
15: F4-C4
16: C4-P4
17: P4-O2
18: -
19: FP2-F8
20: F8-T8
21: T8-P8
22: P8-O2
23: -
24: P7-T7
25: T7-FT9
26: FT9-FT10
27: FT10-T8
28: T8-P8
29: LOC-ROC

--- Channel configuration 2 ---
n_files: 10
n_channels: [28]
files: ['chb12_06.edf', 'chb12_08.edf', 'chb12_09.edf', 'chb12_10.edf', 'chb12_11.edf']
labels:
01: FP1-F7
02: F7-T7
03: T7-P7
04: P7-O1
05: -
06: FP1-F3
07: F3-C3
08: C3-P3
09: P3-O1
10: -
11: FZ-CZ
12: CZ-PZ
13: -
14: FP2-F4
15: F4-C4
16: C4-P4
17: P4-O2
18: -
19: FP2-F8
20: F8-T8
21: T8-P8
22: P8-O2
23: -
24: P7-T7
25: T7-FT9
26: FT9-FT10
27: FT10-T8
28: T8-P8

--- Channel configuration 3 ---
n_files: 2
n_channels: [29]
files: ['chb12_28.edf', 'chb12_29.edf']
labels:
01: F7
02: T7
03: P7
04

## Per-Patient Raw Data Summary

In [9]:
df["has_dup_labels"] = df["n_duplicate_labels"].fillna(0).astype(int) > 0
df["is_flagged_name"] = df["filename_flags"].fillna("") != ""

raw_patient_summary = (
    df.groupby("patient", as_index=False)
    .agg(
        n_edf=("file", "count"),
        n_unique_chan_lists=("sig_ordered", "nunique"),
        n_unique_chan_multisets=("sig_multiset", "nunique"),
        min_channels=("n_channels", "min"),
        max_channels=("n_channels", "max"),
        n_files_w_dup_labels=("has_dup_labels", "sum"),
        n_flagged_files=("is_flagged_name", "sum"),
        fs_sets=("fs_set", lambda x: sorted(set(x))),
    )
)

def classify_channel_layout(row):
    """Classify whether a patient's channel set changes across files.

    channel-set changes: different channels present across files (different multiset)
    order-only changes:  same channels, different ordering across files
    stable:              identical channel list in all files
    """
    if row["n_unique_chan_multisets"] > 1:
        return "channel-set changes"
    if row["n_unique_chan_lists"] > 1 and row["n_unique_chan_multisets"] == 1:
        return "order-only changes"
    return "stable"

raw_patient_summary["layout_class"] = raw_patient_summary.apply(classify_channel_layout, axis=1)


In [10]:
# n_unique_chan_lists is only informative when a patient has order-only changes
# (same channel multiset, different ordering across files). Check if any exist.
has_order_only_changes = (
    (raw_patient_summary["n_unique_chan_lists"] > 1)
    & (raw_patient_summary["n_unique_chan_multisets"] == 1)
).any()

if has_order_only_changes:
    raw_patient_summary["has_order_changes"] = (
        (raw_patient_summary["n_unique_chan_lists"] > 1)
        & (raw_patient_summary["n_unique_chan_multisets"] == 1)
    )

raw_patient_summary = raw_patient_summary.drop(columns=["n_unique_chan_lists"])


In [11]:
raw_patient_summary["%_dup"] = (
    raw_patient_summary["n_files_w_dup_labels"] / raw_patient_summary["n_edf"] * 100
).round(1)
raw_patient_summary["%_flagged"] = (
    raw_patient_summary["n_flagged_files"] / raw_patient_summary["n_edf"] * 100
).round(1)

cols = [
    "patient", "n_edf", "layout_class",
    "n_unique_chan_multisets",
    "min_channels", "max_channels",
    "n_files_w_dup_labels", "%_dup",
    "n_flagged_files", "%_flagged",
    "fs_sets",
]

if "has_order_changes" in raw_patient_summary.columns:
    cols.insert(cols.index("n_unique_chan_multisets") + 1, "has_order_changes")

preview(
    raw_patient_summary[cols].sort_values(
        ["layout_class", "n_files_w_dup_labels", "n_flagged_files", "n_edf"],
        ascending=[True, False, False, False]
    ),
    max_rows=50,
)


,patient,n_edf,layout_class,n_unique_chan_multisets,min_channels,max_channels,n_files_w_dup_labels,%_dup,n_flagged_files,%_flagged,fs_sets
3,chb04,42,channel-set changes,2,23,24,42,100.0,0,0.0,"[(256.0,)]"
14,chb15,40,channel-set changes,2,31,38,40,100.0,0,0.0,"[(256.0,)]"
17,chb18,36,channel-set changes,2,22,28,36,100.0,0,0.0,"[(256.0,)]"
10,chb11,35,channel-set changes,2,23,28,35,100.0,0,0.0,"[(256.0,)]"
12,chb13,33,channel-set changes,3,22,28,33,100.0,0,0.0,"[(256.0,)]"
18,chb19,30,channel-set changes,2,22,28,30,100.0,0,0.0,"[(256.0,)]"
11,chb12,24,channel-set changes,4,28,29,24,100.0,0,0.0,"[(256.0,)]"
16,chb17,21,channel-set changes,2,22,28,21,100.0,21,100.0,"[(256.0,)]"
8,chb09,19,channel-set changes,2,23,24,19,100.0,0,0.0,"[(256.0,)]"
15,chb16,19,channel-set changes,2,22,28,19,100.0,0,0.0,"[(256.0,)]"


## Generate Clean EDFs

Calls `process_patient_to_edf` from `homogenize_signals.py` to produce a clean copy of every
raw EDF under `clean_edfs/`. The raw files in `chb-mit-scalp-eeg-database-1.0.0/` are never modified.

**What the cleaning does:**
- Drops non-EEG and unwanted channels (see *EEG Channel Reference* below for full details)
- Disambiguates duplicate channel labels (`T8-P8` → `T8-P8` + `T8-P8-2`)
- Zero-pads any channel missing in a file so every file within a montage set has a uniform channel list
- Splits patients with mid-recording electrode changes into separate `_set0`, `_set1`, … subfolders
- Writes EDF+ files with seizure annotations embedded as EDF+ annotation records

**Re-running after a code change:**  
Set `FORCE_REPROCESS = True` in the cell below to overwrite existing output files.  
Set it to `False` to skip already-written files and resume an interrupted run.

In [12]:
from homogenize_signals import process_patient_to_edf

# Set FORCE_REPROCESS=True to overwrite existing clean EDFs (e.g. after
# updating the channel-filtering logic in homogenize_signals.py).
# Leave False to skip already-processed files and resume a partial run.
FORCE_REPROCESS = False

CLEAN_EDF_ROOT.mkdir(exist_ok=True)

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset not found: {DATASET_ROOT.resolve()}")

patient_dirs = sorted(
    p for p in DATASET_ROOT.iterdir()
    if p.is_dir() and re.fullmatch(r"chb\d{2}", p.name)
)
patient_ids = [p.name[3:] for p in patient_dirs]

# Run processing, capturing stdout from each patient to parse the summary line
logs: dict[str, str] = {}
buf = io.StringIO()
for pid in tqdm(
    patient_ids,
    desc="Generating clean EDFs",
    unit="patient",
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}  [elapsed: {elapsed}  left: {remaining}]",
    ncols=80,
):
    buf.truncate(0)
    buf.seek(0)
    with contextlib.redirect_stdout(buf):
        process_patient_to_edf(pid, str(DATASET_ROOT), str(CLEAN_EDF_ROOT), force=FORCE_REPROCESS)
    logs[pid] = buf.getvalue()

# Parse the summary line emitted by process_patient_to_edf for each patient
_pat = re.compile(r"processed=(\d+), skipped=(\d+), sets=(\[.*?\])")
rows = []
for pid, log in logs.items():
    m = _pat.search(log)
    if m:
        processed, skipped, sets_str = int(m.group(1)), int(m.group(2)), m.group(3)
    else:
        processed, skipped, sets_str = 0, 0, "?"
    rows.append({"patient": f"chb{pid}", "written": processed, "skipped": skipped, "sets": sets_str})

stats = pd.DataFrame(rows)
total_written = stats["written"].sum()
total_skipped = stats["skipped"].sum()

# Parse the set list string (e.g. "[0, 1, 2]") to count distinct montage sets per patient
stats["n_sets"] = stats["sets"].apply(
    lambda s: len(ast.literal_eval(s)) if s not in ("", "?") else 0
)
multi_set_pats = stats[stats["n_sets"] > 1]["patient"].tolist()

# ── Summary line ──────────────────────────────────────────────────────────────
print(f"✓  {total_written} files written  ({len(patient_ids)} patients)")
if multi_set_pats:
    print(f"   Multi-set patients : {', '.join(multi_set_pats)}")

# ── Flagged table ─────────────────────────────────────────────────────────────
# When FORCE_REPROCESS=False, "skipped" means "already exists" — not an error.
# Errors are only meaningful when force=True (every file should have been written).
has_errors = FORCE_REPROCESS and total_skipped > 0
failures   = stats[stats["skipped"] > 0].copy() if has_errors else pd.DataFrame()
multi_set  = stats[stats["n_sets"] > 1].copy()
flagged    = pd.concat([failures, multi_set]).drop_duplicates("patient")

if flagged.empty:
    print("   All patients: single set, no errors.")
else:
    print()

    def _fmt_sets(n):
        return f"{n} set{'s' if n != 1 else ''}"

    error_patients = set(failures["patient"]) if has_errors else set()

    cols = ["patient", "sets"] if not FORCE_REPROCESS else ["patient", "written", "sets"]
    display_df = flagged[cols].copy()
    display_df["sets"] = flagged["n_sets"].apply(_fmt_sets)
    display_df = display_df.rename(columns={"written": "files written", "sets": "montage sets"})
    if has_errors:
        display_df["errors"] = flagged["skipped"].values

    def _flag_row(row):
        color = "color: #e07b54;" if row["patient"] in error_patients else ""
        return [color] * len(row)

    caption = "Patients with multiple montage sets" + (
        " or processing errors" if has_errors else ""
    )
    styler = (
        display_df.style
        .apply(_flag_row, axis=1)
        .set_caption(caption)
        .set_table_styles([
            {"selector": "caption", "props": [
                ("font-weight", "bold"), ("text-align", "left"), ("margin-bottom", "4px"),
            ]},
            {"selector": "th, td", "props": [
                ("white-space", "nowrap"), ("padding", "3px 12px"),
            ]},
        ])
        .hide(axis="index")
    )
    display(styler)


Generating clean EDFs: 100%|██████████████| 24/24  [elapsed: 00:00  left: 00:00]

✓  0 files written  (24 patients)
   Multi-set patients : chb04, chb09, chb11, chb12, chb13, chb15, chb16, chb17, chb18, chb19



patient,montage sets
chb04,2 sets
chb09,2 sets
chb11,2 sets
chb12,4 sets
chb13,6 sets
chb15,2 sets
chb16,2 sets
chb17,2 sets
chb18,2 sets
chb19,2 sets


## Raw vs Clean: Comparison

What changed between the original EDFs and the clean ones:

**Channel filtering** — the following channel types are removed:
- Display/dummy placeholders: labels `-` and `.`
- Cardiac: any label containing `ECG` or `EKG`
- Device channels: `DUMMY`, `VNS` (vagal nerve stimulator)
- Polygraphy (non-EEG): `LOC-ROC`, `LUE-RAE` and their reverses
- Common-reference montage channels: labels ending in `-CS2` or `-Ref`
- Unipolar/absolute channels: labels with no hyphen (single-electrode names such as `C2`, `CZ`)

After filtering, every patient ends up with the same 23-channel longitudinal bipolar montage.

**Structural changes:**
- Duplicate labels are disambiguated (`T8-P8` → `T8-P8` + `T8-P8-2`)
- Patients with montage changes mid-recording are split into numbered subfolders (`_set0`, `_set1`, …)
- Missing channels within a set are zero-padded so every file has a uniform channel list

In [13]:
def read_clean_edf_header(edf_path: Path) -> dict:
    with pyedflib.EdfReader(str(edf_path)) as f:
        labels = [lbl.strip() for lbl in f.getSignalLabels()]
        fs_list = [f.getSampleFrequency(i) for i in range(f.signals_in_file)]
        duration_s = f.getFileDuration()
        # homogenize_signals stores the montage set index in the EDF recording field
        recording = f.getRecordingAdditional().strip()

    montage_match = re.search(r"montage_set=(\d+)", recording)
    montage_set = int(montage_match.group(1)) if montage_match else 0

    folder = edf_path.parent.name           # e.g. "chb04_set1" or "chb01"
    folder_match = re.match(r"(chb\d{2})", folder)
    patient = folder_match.group(1) if folder_match else "unknown"

    label_counts = Counter(labels)
    return {
        "patient":     patient,
        "file":        edf_path.name,
        "folder":      folder,
        "montage_set": montage_set,
        "n_channels":  len(labels),
        "fs_set":      tuple(sorted(set(fs_list))),
        "duration_s":  duration_s,
        "n_duplicate_labels": sum(1 for v in label_counts.values() if v > 1),
        "labels":      labels,
    }

clean_rows = []
for p in sorted(CLEAN_EDF_ROOT.rglob("*.edf")):
    try:
        clean_rows.append(read_clean_edf_header(p))
    except Exception as e:
        clean_rows.append({"file": p.name, "folder": p.parent.name, "error": repr(e)})

df_clean = pd.DataFrame(clean_rows)

# Align column order with df for consistent display
shared_cols = [c for c in df.columns if c in df_clean.columns]
extra_cols = [c for c in df_clean.columns if c not in shared_cols]
df_clean = df_clean[shared_cols + extra_cols]

print(f"Clean EDF files: {len(df_clean)}  ({df_clean['patient'].nunique()} patients)\n")


def compute_dropped_labels(row) -> list:
    """Return raw channel labels absent from the corresponding clean file.

    Handles duplicates: if a label appears twice in the raw file, each occurrence
    is treated separately (the second gets a -2 suffix, matching homogenize_signals
    renaming logic), so a label is only considered dropped if neither its original
    nor its renamed form appears in the clean set.
    """
    if not isinstance(row["labels"], list) or not isinstance(row["sig_ordered"], str):
        return []
    raw_labels = row["sig_ordered"].split("|")
    clean_set = set(row["labels"])
    seen = {}
    dropped = []
    for lbl in raw_labels:
        count = seen.get(lbl, 0)
        seen[lbl] = count + 1
        unique_form = lbl if count == 0 else f"{lbl}-{count + 1}"
        if unique_form not in clean_set and lbl not in clean_set:
            dropped.append(lbl)
    return sorted(set(dropped))


# Per-file diff: join raw and clean records on filename
file_diff = (
    df[["patient", "file", "n_channels", "n_duplicate_labels", "sig_ordered"]]
    .merge(
        df_clean[["file", "folder", "montage_set", "n_channels", "n_duplicate_labels", "labels"]]
        .rename(columns={
            "n_channels": "clean_n_channels",
            "n_duplicate_labels": "clean_n_duplicate_labels",
        }),
        on="file", how="outer", indicator=True,
    )
)

raw_only   = file_diff[file_diff["_merge"] == "left_only"]   # raw files with no clean counterpart
clean_only = file_diff[file_diff["_merge"] == "right_only"]  # clean files with no raw counterpart (shouldn't happen)
matched    = file_diff[file_diff["_merge"] == "both"].copy()

matched["n_ch_removed"] = matched["n_channels"] - matched["clean_n_channels"]
matched["n_dupes_fixed"] = matched["n_duplicate_labels"] - matched["clean_n_duplicate_labels"]
matched["dropped_labels"] = matched.apply(compute_dropped_labels, axis=1)

print(
    f"Files matched: {len(matched)}   "
    f"Raw-only (skipped by homogenizer): {len(raw_only)}   "
    f"Clean-only: {len(clean_only)}\n"
)
print("Files where channels were removed:", (matched["n_ch_removed"] > 0).sum())

# Per-patient summary of dropped labels — channel counts and dup stats are covered in cell 19
dropped_summary = (
    matched[matched["n_ch_removed"] > 0]
    .groupby("patient", as_index=False)
    .agg(
        n_files=("file", "count"),
        dropped_labels=("dropped_labels", lambda x: sorted(set(lbl for lbls in x for lbl in lbls))),
    )
    .sort_values("patient")
)
display(dropped_summary.style.hide(axis="index"))

Clean EDF files: 686  (24 patients)

Files matched: 686   Raw-only (skipped by homogenizer): 0   Clean-only: 0

Files where channels were removed: 410


patient,n_files,dropped_labels
chb04,36,['ECG']
chb09,18,['VNS']
chb11,34,['-']
chb12,24,"['-', 'EKG1-CHIN', 'LOC-ROC']"
chb13,33,"['-', 'EKG1-EKG2', 'LUE-RAE']"
chb14,26,['-']
chb15,40,"['-', 'CP1-Ref', 'CP2-Ref', 'CP5-Ref', 'CP6-Ref', 'FC1-Ref', 'FC2-Ref', 'FC5-Ref', 'FC6-Ref']"
chb16,19,['-']
chb17,21,['-']
chb18,36,"['-', '.']"


In [14]:
# Per-patient summary: aggregate raw and clean metrics then merge
raw_per_patient = (
    df.groupby("patient", as_index=False)
    .agg(
        n_raw=("file", "count"),
        raw_min_channels=("n_channels", "min"),
        raw_max_channels=("n_channels", "max"),
        raw_dup_files=("n_duplicate_labels", lambda x: (x > 0).sum()),
    )
)

clean_per_patient = (
    df_clean.groupby("patient", as_index=False)
    .agg(
        n_clean=("file", "count"),
        clean_min_channels=("n_channels", "min"),
        clean_max_channels=("n_channels", "max"),
        montage_sets=("montage_set", lambda x: sorted(set(x))),
        clean_dup_files=("n_duplicate_labels", lambda x: (x > 0).sum()),
    )
)

patient_comparison = (
    raw_per_patient
    .merge(clean_per_patient, on="patient", how="outer")
    .sort_values("patient")
)
patient_comparison["n_skipped"]   = patient_comparison["n_raw"] - patient_comparison["n_clean"]
patient_comparison["multi_set"]   = patient_comparison["montage_sets"].apply(
    lambda x: len(x) > 1 if isinstance(x, list) else False
)
patient_comparison["ch_reduced"]  = (
    patient_comparison["raw_max_channels"] - patient_comparison["clean_max_channels"]
)

preview(patient_comparison[[
    "patient",
    "n_raw", "n_clean", "n_skipped",
    "raw_min_channels", "raw_max_channels",
    "clean_min_channels", "clean_max_channels", "ch_reduced",
    "raw_dup_files", "clean_dup_files",
    "montage_sets", "multi_set",
]], max_rows=len(patient_comparison))


,patient,n_raw,n_clean,n_skipped,raw_min_channels,raw_max_channels,clean_min_channels,clean_max_channels,ch_reduced,raw_dup_files,clean_dup_files,montage_sets,multi_set
0,chb01,42,42,0,23,23,23,23,0,42,0,[0],False
1,chb02,36,36,0,23,23,23,23,0,36,0,[0],False
2,chb03,38,38,0,23,23,23,23,0,38,0,[0],False
3,chb04,42,42,0,23,24,23,23,1,42,0,"[0, 1]",True
4,chb05,39,39,0,23,23,23,23,0,39,0,[0],False
5,chb06,18,18,0,23,23,23,23,0,18,0,[0],False
6,chb07,19,19,0,23,23,23,23,0,19,0,[0],False
7,chb08,20,20,0,23,23,23,23,0,20,0,[0],False
8,chb09,19,19,0,23,24,23,23,1,19,0,"[0, 1]",True
9,chb10,25,25,0,23,23,23,23,0,25,0,[0],False


## Summary Report: Raw → Clean Transformation

### Understanding the Report

**Montage Sets (Why multiple sets?)**  
Some patients had electrode placement changes during monitoring. When this happens, the channel configuration changes — either different electrodes are used or they're placed in different locations. The cleaning process detects these changes and splits such patients into separate subfolders (`chb17_set0`, `chb17_set1`, etc.), where each set has a stable, consistent channel layout with zero-padding applied so all files within a set are uniform.

If a patient has only one stable montage they remain in a single folder (e.g. `chb01`) with no `_set` suffix.

**Channels column**  
Shows the channel count range across all files: `raw_min–raw_max → clean_min–clean_max (−N)`.  
If raw and clean are identical the arrow is omitted.  
The reduction comes from dropping non-EEG and non-bipolar channels (see *Raw vs Clean: Comparison* above).

**Final labels column**  
Describes each montage set's channel list relative to the 23-channel standard bipolar set:
- `std only` — the set contains exactly the standard 23 channels
- `std + …` — the set has extra channels beyond the standard 23 (should not appear after re-cleaning)

### Standard channel set

The cell below computes the **standard channel list**: the intersection of all channels present across every patient's clean files. This is the 23-channel bipolar montage described in *EEG Channel Reference*.

The per-patient `final_labels` column in the report expresses each montage set relative to this standard:

- `std only` — the set contains exactly the standard 23 channels (expected for all patients after cleaning)
- `std + …` — the set has channels beyond the standard list (indicates the cleaning filter missed something)
- Multi-set patients show one line per set

### EEG Channel Reference

#### The 10-20 system and bipolar montage

The CHB-MIT recordings use the **International 10-20 system**, the clinical standard for scalp EEG electrode placement. Electrode positions are named by region:

| Prefix | Region | Notes |
|--------|--------|-------|
| **Fp** | Frontopolar | Forehead, above eyes |
| **F** | Frontal | Planning, executive function |
| **C** | Central | Sensorimotor cortex (primary motor + somatosensory) |
| **P** | Parietal | Spatial processing, sensory integration |
| **O** | Occipital | Visual cortex |
| **T** | Temporal | Memory, language, auditory processing |
| **FT** | Fronto-temporal | Between frontal and temporal |
| **Z suffix** | Midline | e.g. FZ, CZ, PZ sit on the sagittal midline |

Odd numbers → left hemisphere; even numbers → right hemisphere.

All channels in this dataset are **bipolar derivations**: the signal is the voltage *difference* between two adjacent electrodes (e.g. `F7-T7` = voltage at F7 minus voltage at T7). This cancels out common-mode noise and highlights local cortical activity between the two electrodes.

---

#### Standard channel set (23 channels)

The 23 channels form a **longitudinal bipolar montage** ("double banana") — six parallel anterior-to-posterior chains that sweep the scalp from forehead to occiput:

```
Left temporal chain  (outer left):   FP1-F7 → F7-T7 → T7-P7 → P7-O1
Left parasagittal chain (inner left): FP1-F3 → F3-C3 → C3-P3 → P3-O1
Right parasagittal chain (inner right): FP2-F4 → F4-C4 → C4-P4 → P4-O2
Right temporal chain (outer right):  FP2-F8 → F8-T8 → T8-P8 → P8-O2
Midline chain:                        FZ-CZ → CZ-PZ
Temporal extension (inferior loop):  P7-T7 → T7-FT9 → FT9-FT10 → FT10-T8
```

The **temporal extension** chain (FT9, FT10) sits below the standard 10-20 grid, over the inferior temporal lobe and anterior temporal tip — a region particularly relevant for **mesial temporal lobe epilepsy**, the most common focal epilepsy type. This is why it appears in an epilepsy dataset but is less common in general clinical EEG.

`T8-P8` and `T8-P8-2` are the same electrode pair that appeared **twice** in some raw files (a duplicate label artifact). After cleaning, the second occurrence is renamed `T8-P8-2` to make channel lists unambiguous.

---

#### Non-standard configurations ("std + ...")

Some patients were recorded with extra or different electrodes, all present *alongside* the standard 23:

| Extra channels | Patient(s) | Type | Explanation |
|----------------|-----------|------|-------------|
| `VNS` | chb09 (set 1) | Stimulus | Vagal Nerve Stimulator pulse channel — records when the implanted device fires, **not** brain activity. |
| `C2`, `CZ`, `T7`, … (no hyphen) | chb12 (set 0) | Unipolar EEG | Single-electrode absolute potentials referenced to an implicit common ref. Same electrode positions as the standard channels but not bipolar derivations. |
| `C3-CS2`, `FP1-CS2`, … | chb12 (set 1) | Common-ref EEG | All standard electrode positions referenced to CS2 (a combined scalp/mastoid reference). The same brain activity looks different waveform-wise compared to bipolar: common-ref preserves absolute amplitudes, bipolar suppresses shared noise. You can *reconstruct* bipolar pairs from them (e.g. C3-P3 ≈ (C3-CS2) − (P3-CS2)), but they are redundant with the already-present bipolar channels. |
| `LOC-ROC` | chb12 (set 3) | Polygraphy (ocular) | Left Outer Canthus → Right Outer Canthus — horizontal eye movement channel (EOG). Not EEG. |
| `LUE-RAE` | chb13 (set 1) | Polygraphy (auricular) | Left Upper Eyelid → Right Auricular Electrode. Not EEG. |
| `FC1-Ref`, `CP1-Ref`, … | chb15 (sets 0–1) | Extra EEG (Ref) | FC (fronto-central) and CP (centro-parietal) electrodes at intermediate 10-20 positions, providing denser coverage over the motor cortex. Referenced to a common ref rather than bipolar. |
| `PZ-OZ` | chb15 (set 1) | Extra EEG (bipolar) | Posterior midline bipolar derivation — Pz to Oz. Covers the occipital region not reached by any of the 23 standard chains. |

---

#### Cleaning decisions

The goal of the synthesis pipeline is **cross-patient generalization**: the same 23-channel feature vector must be valid for every patient, so the synthetic data generator has a fixed, consistent input space. This drives the per-category decisions:

| Category | Decision | Reasoning |
|----------|----------|-----------|
| `VNS` | **Drop entirely** | Not EEG. The stimulator fires in response to (or correlated with) seizures — including it as a model input is **data leakage**. It can be cross-referenced against seizure annotations independently. |
| `LOC-ROC`, `LUE-RAE` | **Drop entirely** | Not EEG. Eye deviation is a clinical ictal sign in some frontal seizures, but it is a *consequence* already visible in the frontal EEG channels. Only present in two patients, and cannot be synthesized with the same approach as scalp EEG. |
| `*-CS2`, unipolar | **Drop entirely** | Redundant with the bipolar channels already in the file. Keeping both would double-count the same brain activity in two different reference schemes. |
| `FC*-Ref`, `CP*-Ref`, `PZ-OZ` | **Drop from primary channels** | These are genuine, non-redundant EEG channels but only present in chb15. Including them in the main feature vector would make chb15 incompatible with all other patients. They are dropped from the primary signal dict to maintain a consistent 23-channel layout across the whole dataset. |

The net result is that **all patients end up with exactly the same 23-channel bipolar montage** after cleaning — a necessary precondition for training a cross-patient synthetic data model.

In [15]:
# Determine the standard channel set: channels present in every patient's clean files
patient_label_union = {}
for patient_id, group in df_clean.groupby("patient"):
    label_sets = [set(lbls) for lbls in group["labels"] if isinstance(lbls, list)]
    patient_label_union[patient_id] = set.union(*label_sets) if label_sets else set()

std_channels = (
    sorted(set.intersection(*patient_label_union.values()))
    if patient_label_union else []
)

# Display standard channels; wrap each name in nowrap so hyphenated channels (e.g. FT9-FT10) never break
channel_spans = ", ".join(
    f'<span style="white-space: nowrap">{ch}</span>'
    for ch in std_channels
)
display(HTML(
    '<p style="font-family: monospace; white-space: normal; '
    'max-width: 1200px; line-height: 1.6;">'
    '<strong>Standard channels (present in every patient):</strong> '
    + channel_spans
    + '</p>'
))


def describe_label_set(labels: tuple) -> str:
    """Describe a channel set relative to the standard channel list."""
    extras = set(labels) - set(std_channels)
    return f"std + {', '.join(sorted(extras))}" if extras else "std only"

def colorize_label_desc(desc: str) -> str:
    """Wrap a label description in a colored span."""
    if "std +" in desc:
        return f'<span style="color: #7ec8e3;">{desc}</span>'
    return f'<span style="font-weight: bold; color: #6fcf7f;">{desc}</span>'


# Build one row per patient for the summary report
report_rows = []
for _, row in patient_comparison.iterrows():
    patient_id = row["patient"]

    # Channel count range: raw → clean (only show arrow when something changed)
    raw_range = f"{int(row['raw_min_channels'])}–{int(row['raw_max_channels'])}"
    clean_range = f"{int(row['clean_min_channels'])}–{int(row['clean_max_channels'])}"
    ch_delta = f" (−{int(row['ch_reduced'])})" if row["ch_reduced"] > 0 else ""
    if raw_range == clean_range and not ch_delta:
        channels_str = raw_range
    else:
        channels_str = f"{raw_range} → {clean_range}{ch_delta}"

    # Duplicate labels present in the raw files for this patient
    patient_raw_files = df[df["patient"] == patient_id]
    all_dups = set()
    for dup_list in patient_raw_files["duplicate_labels"]:
        if isinstance(dup_list, list):
            all_dups.update(dup_list)
    duplicates_str = ", ".join(sorted(all_dups)) if all_dups else "N/A"

    # Channel labels removed during cleaning
    patient_matched_files = matched[matched["patient"] == patient_id]
    all_dropped = set()
    for dropped_list in patient_matched_files["dropped_labels"]:
        if isinstance(dropped_list, list):
            all_dropped.update(dropped_list)
    removed_str = ", ".join(sorted(all_dropped)) if all_dropped else ""

    # Final label sets in the clean files (one per montage configuration)
    patient_clean_files = df_clean[df_clean["patient"] == patient_id]
    unique_final_sets = sorted({
        tuple(lbls)
        for lbls in patient_clean_files["labels"]
        if isinstance(lbls, list)
    })
    if len(unique_final_sets) == 1:
        final_labels_str = colorize_label_desc(describe_label_set(unique_final_sets[0]))
    else:
        final_labels_str = "<br>".join(
            f"set {i}: {colorize_label_desc(describe_label_set(s))}"
            for i, s in enumerate(unique_final_sets)
        )

    # Montage stability
    montage_set_list = row["montage_sets"]
    montage_str = (
        f"Split into {len(montage_set_list)}"
        if isinstance(montage_set_list, list) and len(montage_set_list) > 1
        else "Stable"
    )

    report_rows.append({
        "patient":      patient_id,
        "channels":     channels_str,
        "duplicates":   duplicates_str,
        "removed":      removed_str,
        "final_labels": final_labels_str,
        "montage":      montage_str,
    })

report_df = pd.DataFrame(report_rows)

# Render with HTML so colored spans and <br> line breaks work inside cells
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

print("TRANSFORMATION SUMMARY BY PATIENT")
print("=" * 200)

styler = report_df.style.set_table_attributes('style="width: 100%"')
styler = styler.set_table_styles([
    {"selector": "td:nth-child(1)", "props": [("white-space", "pre")]},
])
styler = styler.set_properties(subset=["channels", "montage"], **{
    "white-space": "nowrap",
    "width": "1px",
})
styler = styler.set_properties(subset=["final_labels"], **{
    "overflow-wrap": "break-word",
})

display(HTML(f'<div style="overflow-x: auto;">{styler.to_html(escape=False)}</div>'))

TRANSFORMATION SUMMARY BY PATIENT


,patient,channels,duplicates,removed,final_labels,montage
0,chb01,23–23,T8-P8,,std only,Stable
1,chb02,23–23,T8-P8,,std only,Stable
2,chb03,23–23,T8-P8,,std only,Stable
3,chb04,23–24 → 23–23 (−1),T8-P8,ECG,std only,Split into 2
4,chb05,23–23,T8-P8,,std only,Stable
5,chb06,23–23,T8-P8,,std only,Stable
6,chb07,23–23,T8-P8,,std only,Stable
7,chb08,23–23,T8-P8,,std only,Stable
8,chb09,23–24 → 23–23 (−1),T8-P8,VNS,std only,Split into 2
9,chb10,23–23,T8-P8,,std only,Stable
